# 02 — Data Architecture: Requirements, Schema, Field Sources & Simulated Sample

**Project:** Booking Cancellation Decision Support System — City Hotel, Lisbon (BA Advanced, Group 2)
**Input:** `data/processed/bookings_city_clean.csv` (from notebook 01) · `config/assumptions.yaml` · `config/column_groups.json`

| Section | Task | Deliverable |
|---|---|---|
| 1 | Which data the decisions need | Data requirements |
| 2 | Tables, keys and relationships | Data architecture / schema (`sql/schema.sql`) |
| 3 | Build the synthetic & derived tables | `data/synthetic/*.csv` |
| 4 | Consistency checks | Validation |
| 5 | Where every field comes from | Data sources & features (`docs/data_dictionary.csv`) |
| 6 | Real + simulated rows, clearly marked | Example data extract |
| 7 | Export tables, load into PostgreSQL | `sql/load.sql` |
| 8 | Integration & privacy | Integration and privacy considerations |

> **Labelling rule.** Every field is one of: **REAL** (from the hotel PMS dataset), **DERIVED** (computed only from real data), **SYNTHETIC** (simulated from `config/assumptions.yaml`), or **SYSTEM** (written later by the model, rule engine or n8n). Synthetic tables carry `is_synthetic = True`, and synthetic numbers must be presented as *"simulated, based on stated assumptions"*.

## 0. Setup

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

ROOT = Path.cwd()
while not (ROOT / "hotel_bookings.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CLEAN_PATH = ROOT / "data" / "processed" / "bookings_city_clean.csv"
SYN_DIR = ROOT / "data" / "synthetic"; SYN_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR = ROOT / "docs"; DOCS_DIR.mkdir(exist_ok=True)

A = yaml.safe_load(open(ROOT / "config" / "assumptions.yaml"))
GROUPS = json.load(open(ROOT / "config" / "column_groups.json"))
rng = np.random.default_rng(A["meta"]["random_seed"])

bk = pd.read_csv(CLEAN_PATH, parse_dates=["arrival_date", "booking_date", "reservation_status_date"])
print(f"bookings: {bk.shape[0]:,} rows x {bk.shape[1]} cols")

bookings: 79,330 rows x 39 cols


## 1. Data requirements

The decisions in the PDMT framing determine the data we need. Each data need maps to one table in the architecture.

| Decision (who) | Data needed | Table | Source |
|---|---|---|---|
| Predict cancellation risk (revenue mgr) | Booking attributes, guest history, channel, price, lead time | `bookings` | REAL |
| Estimate revenue affected (finance) | Room revenue, room cost per night, channel commission, cancellation handling cost | `bookings` + `fin_booking` | REAL + SYNTHETIC |
| Follow-up / reminders (reservations, CX) | Whether, when and how the guest was contacted, and the response | `cx_contact` | SYNTHETIC |
| Tailor messages, CX evaluation | Cancellation reason (text), post-stay satisfaction | `cx_feedback` | SYNTHETIC |
| Capacity allocation by room type (revenue mgr, ops) | Sellable rooms, rooms sold and cancelled per date × room type | `dim_room_type`, `ops_inventory_daily` | DERIVED + SYNTHETIC |
| Resale of cancelled rooms (revenue mgr) | Which cancelled room-nights were resold, at what price | `ops_inventory_daily`, `fin_booking.rebooking_revenue` | SYNTHETIC |
| Price adjustment (revenue mgr) | ADR history per date × room type, occupancy | `bookings`, `ops_inventory_daily` | REAL + DERIVED |
| Approve and execute actions (staff, n8n) | Risk scores, proposed actions, approvals, execution log | `risk_scores`, `action_plan`, `interventions` | SYSTEM (later stages) |
| Keep assumptions traceable | Every simulated parameter with basis and source | — (kept in `config/assumptions.yaml`) | CONFIG |

## 2. Schema and relationships

The full DDL, with types, primary/foreign keys and CHECK constraints, is in **`sql/schema.sql`** (PostgreSQL, schema `hotel`).

```
                        ┌──────────────────┐
                        │  dim_room_type   │ PK room_type
                        └───┬──────────┬───┘
            reserved/assigned│          │room_type
┌──────────────┐  1:1  ┌─────▼──────┐   │      ┌──────────────────────────┐
│ cx_contact   ├───────┤  bookings  │   └──────► ops_inventory_daily      │
│ cx_feedback  ├───────┤ PK booking_│  stay    │ PK (stay_date, room_type)│
│ fin_booking  ├───────┤    id      ├ nights ─►│ (date × room type)       │
└──────────────┘       └─────┬──────┘          └────────────┬─────────────┘
                             │1:n                           │1:n
                     ┌───────▼──────┐   ┌───────────────────▼──┐   ┌───────────────┐
                     │ risk_scores  │   │ action_plan          ├──►│ interventions │
                     └──────────────┘   │ booking- or date-level│1:n└───────────────┘
                                        └──────────────────────┘
```

| Table | Layer | Grain | Primary key | Foreign keys | Rows (this build) |
|---|---|---|---|---|---|
| `dim_room_type` | DERIVED | room type | `room_type` | — | 9 |
| `bookings` | REAL (+ derived cols) | booking | `booking_id` | `reserved_room_type`, `assigned_room_type` → `dim_room_type` | 79,330 |
| `cx_contact` | SYNTHETIC | booking | `booking_id` | → `bookings` | 79,330 |
| `cx_feedback` | SYNTHETIC | booking | `booking_id` | → `bookings` | 79,330 |
| `fin_booking` | SYNTHETIC | booking | `booking_id` | → `bookings` | 79,330 |
| `ops_inventory_daily` | DERIVED + SYNTHETIC | date × room type | `(stay_date, room_type)` | `room_type` → `dim_room_type` | 7,227 |
| `risk_scores`, `action_plan`, `interventions`, `pilot_assignment` | SYSTEM | score / action / execution / randomised pilot arm | surrogate or composite keys | → `bookings`, `ops_inventory_daily`, `action_plan` | 0 (filled later) |

**Join paths.** Booking-level tables join 1:1 on `booking_id`. A booking joins to `ops_inventory_daily` through its **stay nights**: `stay_date ∈ [arrival_date, arrival_date + total_nights)`, using `assigned_room_type` for rooms sold and `reserved_room_type` for cancellations. Views `v_booking_360` and `v_ops_daily_hotel` in `schema.sql` give analysis-ready joins.

## 3. Build the derived and synthetic tables

### 3.1 Stay-night expansion (DERIVED)

In [2]:
def expand_nights(df):
    d = df.loc[df.total_nights > 0].copy()
    d = d.loc[d.index.repeat(d.total_nights)]
    d["stay_date"] = d.arrival_date + pd.to_timedelta(d.groupby("booking_id").cumcount(), unit="D")
    return d

nights = expand_nights(bk[["booking_id", "arrival_date", "total_nights", "is_canceled", "reservation_status",
                           "reserved_room_type", "assigned_room_type", "adr_clean"]])
sold_n = nights[nights.is_canceled == 0]
canc_n = nights[nights.reservation_status == "Canceled"]
print(f"stay nights: {len(nights):,} | sold: {len(sold_n):,} | cancelled (excl. no-show): {len(canc_n):,}")

stay nights: 236,256 | sold: 135,153 | cancelled (excl. no-show): 97,600


### 3.2 `dim_room_type`: capacity (DERIVED)

The dataset has no room inventory. Capacity per room type = the **peak number of occupied rooms** of that type on any night between Sep 2015 and Aug 2017 (partial windows excluded). That is the smallest capacity consistent with the real data. We then sanity-check it against the Lisbon 2016 occupancy benchmarks (a city benchmark cannot identify this hotel's true capacity).

> The peak uses realised stays up to Aug 2017, i.e. it includes the 2017 test period. Capacity is therefore **ops / scenario input only** and never feeds a classifier feature (see `excluded_features` in `config/column_groups.json`). In scenarios, capacity may only be scaled **up** while observed demand is held fixed (`assumptions.yaml`: `room_capacity_by_type.constraint`).

In [3]:
CAP_WINDOW = A["operations"]["room_capacity_by_type"]["window"]     # complete months only
occ_type = (sold_n.groupby(["stay_date", "assigned_room_type"]).size().unstack(fill_value=0)
            .loc[CAP_WINDOW[0]:CAP_WINDOW[1]])
all_types = sorted(set(bk.reserved_room_type) | set(bk.assigned_room_type))
non_sellable = set(A["operations"]["non_sellable_room_types"]["value"])

dim_room_type = pd.DataFrame({"room_type": all_types})
dim_room_type["capacity_rooms"] = dim_room_type.room_type.map(occ_type.max()).fillna(0).astype(int)
dim_room_type["is_sellable"] = ~dim_room_type.room_type.isin(non_sellable) & (dim_room_type.capacity_rooms > 0)
dim_room_type.loc[~dim_room_type.is_sellable, "capacity_rooms"] = 0
dim_room_type["capacity_source"] = "derived: peak occupied rooms per assigned_room_type, 2015-09..2017-08"
CAP = dim_room_type.set_index("room_type").capacity_rooms
TOTAL_ROOMS = int(CAP.sum())

hotel_occ_2016 = occ_type.loc["2016"].sum(axis=1).mean() / TOTAL_ROOMS
bm = A["benchmarks"]
print(f"Total sellable rooms: {TOTAL_ROOMS}")
print(f"Implied average occupancy 2016: {hotel_occ_2016:.1%} "
      f"(Lisbon 2016 benchmark: 5* {bm['lisbon_occupancy_5star_2016']['value']:.1%}, 4* {bm['lisbon_occupancy_4star_2016']['value']:.1%})")
dim_room_type

Total sellable rooms: 252
Implied average occupancy 2016: 71.3% (Lisbon 2016 benchmark: 5* 68.7%, 4* 79.1%)


,room_type,capacity_rooms,is_sellable,capacity_source
0,A,146,True,derived: peak occupied rooms per assigned_room...
1,B,15,True,derived: peak occupied rooms per assigned_room...
2,C,3,True,derived: peak occupied rooms per assigned_room...
3,D,57,True,derived: peak occupied rooms per assigned_room...
4,E,10,True,derived: peak occupied rooms per assigned_room...
5,F,8,True,derived: peak occupied rooms per assigned_room...
6,G,5,True,derived: peak occupied rooms per assigned_room...
7,K,8,True,derived: peak occupied rooms per assigned_room...
8,P,0,False,derived: peak occupied rooms per assigned_room...


### 3.3 `ops_inventory_daily`: rooms sold and cancelled per date × room type (DERIVED)

In [4]:
dates = pd.date_range(bk.arrival_date.min(), nights.stay_date.max(), freq="D")
grid = pd.MultiIndex.from_product([dates, all_types], names=["stay_date", "room_type"])

ops = pd.DataFrame(index=grid)
ops["available_rooms"] = ops.index.get_level_values("room_type").map(CAP).astype(int)
ops["rooms_sold"] = sold_n.groupby(["stay_date", "assigned_room_type"]).size().reindex(grid, fill_value=0)
ops["rooms_cancelled"] = canc_n.groupby(["stay_date", "reserved_room_type"]).size().reindex(grid, fill_value=0)
ops["occupancy_rate"] = (ops.rooms_sold / ops.available_rooms.replace(0, np.nan)).round(4)
d = ops.index.get_level_values("stay_date")
ops["is_partial_period"] = (d < CAP_WINDOW[0]) | (d > CAP_WINDOW[1])
assert (ops.rooms_sold <= ops.available_rooms).all()
print(f"ops_inventory_daily: {len(ops):,} rows (date x room type)")

ops_inventory_daily: 7,227 rows (date x room type)


### 3.4 Finance: room cost and channel commission (SYNTHETIC)

- `room_cost_per_night` = `rooms_cost_ratio` × median ADR of (reserved room type, arrival month) × U(1 ± noise).
- `channel_commission_rate` = rate by `market_segment`, overridden for `distribution_channel == "GDS"`.

In [5]:
F = A["finance"]
ref_adr = (bk[bk.adr_clean > 0].groupby(["reserved_room_type", "arrival_month_num"]).adr_clean.median())
type_adr = bk[bk.adr_clean > 0].groupby("reserved_room_type").adr_clean.median()
base_adr = pd.Series(list(zip(bk.reserved_room_type, bk.arrival_month_num))).map(ref_adr)
base_adr = base_adr.fillna(bk.reserved_room_type.map(type_adr)).fillna(bk.adr_clean[bk.adr_clean > 0].median())

noise = F["cost_noise"]["value"]
fin = pd.DataFrame({"booking_id": bk.booking_id})
fin["room_cost_per_night"] = (F["rooms_cost_ratio"]["value"] * base_adr.values
                              * rng.uniform(1 - noise, 1 + noise, len(bk))).round(2)
comm = bk.market_segment.map(F["commission_rate_by_segment"]["value"]).fillna(0.0)
comm = comm.where(bk.distribution_channel != "GDS", F["gds_commission_rate"]["value"])
fin["channel_commission_rate"] = comm.round(3).values

### 3.5 Resale of cancelled rooms (SYNTHETIC): `resold_after_cancellation` and `rebooking_revenue`

One simulation feeds both tables, so they are consistent:
1. For each **cancelled** room-night (no-shows excluded, since a no-show room can't be resold), P(resold) depends on **days of notice** (arrival − cancellation date) and **demand** (occupancy of that date × room type ≥ threshold).
2. Resold rooms are part of the real rooms sold, so per date × room type: `resold ≤ min(rooms_cancelled, rooms_sold)`. Any excess draws are removed at random.
3. Resale price = median ADR of rooms sold that date × room type (fallback: month × type) × `resale_price_factor`.
4. `fin_booking` keeps both the resale revenue and the number of resold room-nights per booking, so notebook 03 can deduct the room cost and channel commission of the replacement sale (gross resale revenue is **not** the recovered contribution).

> These are **simulated** outputs of team assumptions, bounded by the real rooms sold. They are not observed resale, and they say nothing about what an intervention would add.

In [6]:
O = A["operations"]
P = O["resale_probability"]["value"]
thr = O["high_demand_occupancy_threshold"]["value"]

cn = canc_n.merge(bk[["booking_id", "reservation_status_date"]], on="booking_id")
cn["notice_days"] = (bk.set_index("booking_id").arrival_date.reindex(cn.booking_id).values
                     - cn.reservation_status_date.values).astype("timedelta64[D]").astype(int)
cn["bucket"] = np.select([cn.notice_days >= 30, cn.notice_days >= 8], ["notice_30_plus", "notice_8_29"], "notice_0_7")
occ_lookup = ops["occupancy_rate"].fillna(0)
cn["occ"] = occ_lookup.reindex(pd.MultiIndex.from_arrays([cn.stay_date, cn.reserved_room_type])).values
cn["demand"] = np.where(cn.occ >= thr, "high_demand", "normal")
cn["p_resale"] = [P[b][d] for b, d in zip(cn.bucket, cn.demand)]
cn.loc[cn.reserved_room_type.map(CAP).fillna(0).eq(0), "p_resale"] = 0.0
cn["resold"] = rng.random(len(cn)) < cn.p_resale

# Cap: resold <= rooms_sold for that date x room type
cn["rand"] = rng.random(len(cn))
cn = cn.sort_values("rand")
cn["rank_in_cell"] = cn[cn.resold].groupby(["stay_date", "reserved_room_type"]).cumcount()
cell_sold = ops["rooms_sold"].reindex(pd.MultiIndex.from_arrays([cn.stay_date, cn.reserved_room_type])).values
cn.loc[cn.resold & (cn.rank_in_cell >= cell_sold), "resold"] = False

# Resale price
sold_price = sold_n[sold_n.adr_clean > 0].groupby(["stay_date", "assigned_room_type"]).adr_clean.median()
month_price = (sold_n[sold_n.adr_clean > 0].assign(m=lambda t: t.stay_date.dt.month)
               .groupby(["m", "assigned_room_type"]).adr_clean.median())
price = sold_price.reindex(pd.MultiIndex.from_arrays([cn.stay_date, cn.reserved_room_type])).values
fallback = month_price.reindex(pd.MultiIndex.from_arrays([cn.stay_date.dt.month, cn.reserved_room_type])).values
cn["resale_adr"] = np.where(np.isnan(price), fallback, price)
cn["resale_adr"] = cn.resale_adr.fillna(type_adr.median()) * O["resale_price_factor"]["value"]
cn["resale_revenue"] = np.where(cn.resold, cn.resale_adr, 0.0)

ops["resold_after_cancellation"] = (cn[cn.resold].groupby(["stay_date", "reserved_room_type"]).size()
                                    .reindex(ops.index, fill_value=0))
reb = cn.groupby("booking_id").resale_revenue.sum().round(2)
fin["rebooking_revenue"] = np.where(bk.reservation_status.eq("Canceled"),
                                    bk.booking_id.map(reb).fillna(0.0), np.nan)
# resold room-nights per booking: lets notebook 03 cost the resale (room cost, commission) on the same basis
reb_n = cn.groupby("booking_id").resold.sum()
fin["rebooked_room_nights"] = (bk.booking_id.map(reb_n).fillna(0).astype(int).astype("Int64")
                               .where(bk.reservation_status.eq("Canceled")))

print(f"Cancelled room-nights: {len(cn):,} | resold: {cn.resold.sum():,} ({cn.resold.mean():.1%})")
cn.groupby(["bucket", "demand"]).resold.agg(["size", "mean"]).round(3)

Cancelled room-nights: 97,600 | resold: 49,491 (50.7%)


size   mean
bucket         demand                   
notice_0_7     high_demand   5117  0.202
               normal        5747  0.095
notice_30_plus high_demand  45560  0.679
               normal       20714  0.433
notice_8_29    high_demand  10684  0.493
               normal        9778  0.283

### 3.6 CX: contact attempts and responses (SYNTHETIC)

This simulates an **assumed** existing practice of **manual pre-arrival confirmations** that are not prioritised by risk (team assumption, to validate with the hotel; the public dataset has no contact log). It gives the follow-up workflow a historical log to design from.
- A contact is planned N days before arrival (never before the booking date). If the booking was **cancelled before** the planned contact, no contact happens.
- The response depends on the outcome. This is why `customer_response` must **never** be a model feature (README §5.3).

> ⚠️ **Do not use this table to estimate the effect of contacting guests.** Contact has **no causal effect** in this simulation, and bookings cancelled before the planned contact date are never contacted. A naive comparison therefore shows contacted bookings cancelling far less than non-contacted ones (survivorship / immortal-time bias), which is an artefact of the generator. Intervention uplift must come from an explicit scenario assumption and, in a real deployment, a pilot with a holdout group.

In [7]:
C = A["cx"]
cp = C["contact_probability"]["value"]
p_contact = (cp["base"] + cp["lead_time_over_30"] * (bk.lead_time > C["contact_probability"]["lead_time_threshold_days"])
             + cp["groups_or_corporate"] * bk.market_segment.isin(["Groups", "Corporate"])
             ).clip(0, C["contact_probability"]["max_probability"])
planned = rng.random(len(bk)) < p_contact

cd = C["contact_days_before_arrival"]["value"]
offset = rng.choice(cd["days"], size=len(bk), p=cd["weights"])
contact_date = (bk.arrival_date - pd.to_timedelta(offset, unit="D"))
contact_date = contact_date.where(contact_date > bk.booking_date, bk.booking_date + pd.Timedelta(days=1))
contact_date = contact_date.where(contact_date <= bk.arrival_date, bk.arrival_date)
cancelled_first = bk.reservation_status.eq("Canceled") & (bk.reservation_status_date < contact_date)
attempted = planned & ~cancelled_first

mix = C["contact_channel_mix"]["value"]
high_value = bk.gross_value >= bk.gross_value.quantile(C["contact_channel_mix"]["high_value_quantile"])
special = bk.market_segment.isin(["Groups", "Corporate"]) | high_value
ch_default = rng.choice(list(mix["default"]), size=len(bk), p=list(mix["default"].values()))
ch_special = rng.choice(list(mix["groups_corporate_or_high_value"]), size=len(bk),
                        p=list(mix["groups_corporate_or_high_value"].values()))
channel = np.where(special, ch_special, ch_default)

R = C["response_probability"]["value"]
def draw_response(outcome, ch):
    p = dict(R[outcome])
    if ch == "phone":
        p["no_response"] *= R["phone_no_response_multiplier"]
    s = sum(p.values())
    return rng.choice(list(p), p=[v / s for v in p.values()])
response = [draw_response("cancelled" if c == 1 else "stayed", ch) if a else None
            for c, ch, a in zip(bk.is_canceled, channel, attempted)]

cx_contact = pd.DataFrame({
    "booking_id": bk.booking_id,
    "contact_attempted": attempted.values,
    "contact_channel": np.where(attempted, channel, None),
    "contact_date": contact_date.where(attempted).dt.date,
    "customer_response": response,
    "is_synthetic": True,
})
print(f"Contacted: {attempted.mean():.1%} of bookings")
pd.crosstab(cx_contact.customer_response, bk.is_canceled, normalize="columns").round(3)

Contacted: 30.1% of bookings


is_canceled,0,1
customer_response,,
confirmed,0.767,0.214
no_response,0.136,0.516
requested_change,0.097,0.270


### 3.7 Finance: cancellation handling cost (SYNTHETIC)

In [8]:
mins = F["handling_minutes"]["value"]
minutes = cx_contact.contact_channel.map(mins).fillna(mins["no_contact"])
fin["cancellation_handling_cost"] = np.where(
    bk.is_canceled.eq(1), (minutes * F["staff_cost_per_hour"]["value"] / 60).round(2), np.nan)
fin["is_synthetic"] = True

### 3.8 CX: satisfaction and cancellation reasons (SYNTHETIC)

- `satisfaction_score` (1–5) only for **Check-Out** bookings, adjusted by observed stay attributes.
- `cancellation_reason_category` + `cancellation_reason_text` only for **Canceled** bookings (no-shows give no reason). About 30% are `not_provided`. The text comes from templates with variation, so TF-IDF + clustering has structure to find and the category can serve as ground truth for evaluating the clusters. This **demonstrates the text pipeline**; it is not evidence of real customer reasons.

In [9]:
S = C["satisfaction"]["value"]
month_med = bk.groupby("arrival_month_num").adr_clean.transform("median")
mu = (S["base_mean"]
      + S["room_type_changed"] * (bk.reserved_room_type != bk.assigned_room_type)
      + S["special_requests"] * (bk.total_of_special_requests > 0)
      + S["repeated_guest"] * bk.is_repeated_guest
      + S["waiting_list"] * (bk.days_in_waiting_list > 0)
      + S["adr_above_month_median_premium"] * (bk.adr_clean > S["adr_premium_multiplier"] * month_med))
score = np.clip(np.rint(rng.normal(mu, S["sd"])), 1, 5)
satisfaction = pd.Series(np.where(bk.reservation_status.eq("Check-Out"), score, np.nan)).astype("Int64")
satisfaction.value_counts(normalize=True).sort_index().round(3)

1    0.001
2     0.03
3    0.227
4    0.459
5    0.283
Name: proportion, dtype: Float64

In [10]:
CR = C["cancellation_reason"]["value"]
cats = list(CR["base_mix"])
base_w = np.array([CR["base_mix"][c] for c in cats])
seg_adr_med = bk.groupby("market_segment").adr_clean.transform("median")

def reason_weights(row):
    w = base_w.copy()
    boosts = []
    if row.market_segment == "Groups": boosts.append(CR["boosts"]["groups_segment"])
    if row.market_segment == "Online TA" and row.adr_clean > row.seg_med: boosts.append(CR["boosts"]["online_ta_high_adr"])
    if row.lead_time > CR["lead_time_boost_threshold_days"]: boosts.append(CR["boosts"]["lead_time_over_180"])
    if row.is_duplicate == 1: boosts.append(CR["boosts"]["duplicate_row"])
    for b in boosts:
        for k, v in b.items(): w[cats.index(k)] *= v
    return w / w.sum()

TEMPLATES = {
    "change_of_plans": ["Our travel plans changed and we can no longer make these dates.",
        "Trip postponed, we will book again later.", "Change of plans, the trip to Lisbon is cancelled.",
        "We had to reschedule our holiday.", "Work schedule changed so I cannot travel on these dates.",
        "Family plans changed, sorry.", "The event we were travelling for was moved to another date."],
    "found_better_price": ["Found a cheaper hotel nearby.", "Got a better rate on another website.",
        "The price is too high compared with other offers.", "Booked a different hotel with a better deal.",
        "Found a lower price for the same dates elsewhere.", "Another platform offered a discount."],
    "duplicate_booking": ["Booked twice by mistake.", "This is a duplicate reservation, please cancel one.",
        "We made the same booking through two websites.", "Double booking, keeping the other reservation.",
        "Accidentally reserved the room again."],
    "travel_restriction_or_illness": ["Flight was cancelled.", "I am sick and cannot travel.",
        "A family member is ill, we have to stay home.", "Visa was not approved in time.",
        "Our flight was rescheduled and we arrive on another day.", "Medical emergency, cannot travel."],
    "group_size_change": ["The group is smaller than planned, cancelling some rooms.",
        "Fewer participants registered for the tour.", "Group size reduced, rooms no longer needed.",
        "The conference delegation was cut, releasing the block.", "Tour operator released unsold rooms."],
    "other": ["Personal reasons.", "No longer needed.", "Prefer not to say.",
        "We decided to stay with friends instead.", "Changed destination."],
}
PREFIX = ["", "", "", "Hello, ", "Hi, ", "Dear hotel, ", "Sorry, "]
SUFFIX = ["", "", "", " Thank you.", " Thanks.", " Regards.", " Please confirm the cancellation."]

def make_text(cat):
    t = rng.choice(TEMPLATES[cat])
    p = rng.choice(PREFIX)
    t = p + (t[0].lower() + t[1:] if p else t)
    t = t + rng.choice(SUFFIX)
    return t.lower() if rng.random() < CR["text_lowercase_probability"] else t

is_canc = bk.reservation_status.eq("Canceled")
tmp = bk.loc[is_canc, ["market_segment", "adr_clean", "lead_time", "is_duplicate"]].assign(seg_med=seg_adr_med[is_canc])
reason_cat = pd.Series(None, index=bk.index, dtype=object)
reason_txt = pd.Series(None, index=bk.index, dtype=object)
for idx, row in tmp.iterrows():
    if rng.random() < CR["not_provided_share"]:
        reason_cat[idx] = "not_provided"
    else:
        c = cats[rng.choice(len(cats), p=reason_weights(row))]
        reason_cat[idx] = c
        reason_txt[idx] = make_text(c)

cx_feedback = pd.DataFrame({
    "booking_id": bk.booking_id,
    "satisfaction_score": satisfaction.values,
    "cancellation_reason_category": reason_cat.values,
    "cancellation_reason_text": reason_txt.values,
    "is_synthetic": True,
})
display(cx_feedback.cancellation_reason_category.value_counts(normalize=True).round(3).to_frame("share"))
cx_feedback.dropna(subset=["cancellation_reason_text"]).sample(8, random_state=A["meta"]["display_sample_seed"])[["cancellation_reason_category", "cancellation_reason_text"]]

,share
cancellation_reason_category,
not_provided,0.301
change_of_plans,0.223
found_better_price,0.137
other,0.112
duplicate_booking,0.106
group_size_change,0.063
travel_restriction_or_illness,0.057


,cancellation_reason_category,cancellation_reason_text
2071,other,"Hi, personal reasons. Please confirm the cance..."
12260,group_size_change,"Sorry, fewer participants registered for the t..."
10591,found_better_price,"Sorry, the price is too high compared with oth..."
26129,found_better_price,Another platform offered a discount. Regards.
24638,other,We decided to stay with friends instead. Pleas...
21707,change_of_plans,We had to reschedule our holiday. Please confi...
20963,found_better_price,Found a lower price for the same dates elsewhere.
36803,group_size_change,"Dear hotel, the conference delegation was cut,..."


## 4. Consistency checks

In [11]:
ops_out = ops.reset_index()
ops_out["is_synthetic"] = True
checks = {
    "1:1 booking tables cover every booking": all(len(t) == len(bk) and t.booking_id.is_unique
                                                   for t in [cx_contact, cx_feedback, fin]),
    "satisfaction only for Check-Out": cx_feedback.satisfaction_score.notna().eq(bk.reservation_status.eq("Check-Out")).all(),
    "reason only for Canceled": cx_feedback.cancellation_reason_category.notna().eq(bk.reservation_status.eq("Canceled")).all(),
    "channel/response only if contacted": (cx_contact.contact_channel.notna() == cx_contact.contact_attempted).all()
                                          and (cx_contact.customer_response.notna() == cx_contact.contact_attempted).all(),
    "no contact after a cancellation": not (bk.reservation_status.eq("Canceled") & cx_contact.contact_attempted
                                            & (pd.to_datetime(cx_contact.contact_date) > bk.reservation_status_date)).any(),
    "contact between booking and arrival": ((pd.to_datetime(cx_contact.contact_date) >= bk.booking_date)
                                            & (pd.to_datetime(cx_contact.contact_date) <= bk.arrival_date))[cx_contact.contact_attempted].all(),
    "handling cost only if is_canceled": fin.cancellation_handling_cost.notna().eq(bk.is_canceled.eq(1)).all(),
    "rebooking revenue only for Canceled": fin.rebooking_revenue.notna().eq(bk.reservation_status.eq("Canceled")).all(),
    "rebooked nights only for Canceled": fin.rebooked_room_nights.notna().eq(bk.reservation_status.eq("Canceled")).all(),
    "rebooking revenue > 0 <=> rebooked nights > 0": ((fin.rebooking_revenue.fillna(0) > 0)
                                                      == (fin.rebooked_room_nights.fillna(0) > 0)).all(),
    "rebooked nights <= booked nights": (fin.rebooked_room_nights.fillna(0) <= bk.total_nights).all(),
    "rooms_sold <= available_rooms": (ops.rooms_sold <= ops.available_rooms).all(),
    "resold <= rooms_cancelled": (ops.resold_after_cancellation <= ops.rooms_cancelled).all(),
    "resold <= rooms_sold": (ops.resold_after_cancellation <= ops.rooms_sold).all(),
    "ops resold == booking-level resold nights": int(ops.resold_after_cancellation.sum()) == int(cn.resold.sum())
                                                 == int(fin.rebooked_room_nights.sum()),
}
res = pd.Series(checks, name="passed").to_frame()
assert res.passed.all(), res[~res.passed]
res

,passed
1:1 booking tables cover every booking,True
satisfaction only for Check-Out,True
reason only for Canceled,True
channel/response only if contacted,True
no contact after a cancellation,True
contact between booking and arrival,True
handling cost only if is_canceled,True
rebooking revenue only for Canceled,True
rebooked nights only for Canceled,True
rebooking revenue > 0 <=> rebooked nights > 0,True


In [12]:
# Headline numbers (simulated, based on stated assumptions)
canc_value = bk.loc[bk.reservation_status.eq("Canceled"), "gross_value"].sum()
pd.Series({
    "Cancelled gross value (EUR, real)": round(canc_value),
    "Rebooking revenue (EUR, simulated, gross)": round(fin.rebooking_revenue.sum()),
    "Resold / cancelled room-nights (simulated, no-shows excluded)": f"{int(fin.rebooked_room_nights.sum()):,} / {len(cn):,}",
    "Gross resale revenue / cancelled gross value (simulated)": f"{fin.rebooking_revenue.sum() / canc_value:.1%}",
    "Cancellation handling cost (EUR, simulated)": round(fin.cancellation_handling_cost.sum()),
    "Commission paid on stayed bookings (EUR, simulated)": round((bk.gross_value * fin.channel_commission_rate)[bk.is_canceled == 0].sum()),
    "Avg occupancy Sep-15..Aug-17 (derived)": f"{ops.loc[~ops.is_partial_period, 'rooms_sold'].sum() / ops.loc[~ops.is_partial_period, 'available_rooms'].sum():.1%}",
}).to_frame("value")

,value
"Cancelled gross value (EUR, real)",10518490
"Rebooking revenue (EUR, simulated, gross)",4832440
"Resold / cancelled room-nights (simulated, no-shows excluded)","49,491 / 97,600"
Gross resale revenue / cancelled gross value (simulated),45.9%
"Cancellation handling cost (EUR, simulated)",42864
"Commission paid on stayed bookings (EUR, simulated)",2183403
Avg occupancy Sep-15..Aug-17 (derived),70.8%


## 5. Data sources and features (field-level)

Each field gets a **source type** (REAL / DERIVED / SYNTHETIC / SYSTEM), a **source detail** (the original column, the derivation, or the assumption key in `config/assumptions.yaml`), a plain-language **description**, its **unit or domain**, a logical **dtype**, a **null rule**, its **key role**, **when the business would need it** (`needed_at`), **what this extract actually holds** (`in_dataset_as`) and its **use**. `needed_at` and `in_dataset_as` are kept apart on purpose: the public extract is a near-arrival snapshot, so a field the business would read at booking entry is usually *not* stored as its booking-time value. Meanings of the original PMS fields follow Antonio, de Almeida & Nunes (2019, Data in Brief). The dictionary is exported to `docs/data_dictionary.csv`. Engineered model features are documented separately in `docs/feature_availability.csv` (notebook 03).

In [13]:
REAL_PMS = "Hotel PMS extract (Antonio et al., 2019), City Hotel H2"
rows = []
def add(table, col, stype, detail, use=""):
    rows.append({"table": table, "column": col, "source_type": stype, "source_detail": detail, "use": use})

derived_detail = {
    "booking_id": "sequential ID 1..79,330 assigned after filtering to City Hotel",
    "booking_date": "arrival_date - lead_time", "arrival_date": "year + month + day columns",
    "has_agent": "agent not null", "has_company": "company not null",
    "dup_group_size": "bookings sharing the same 21 reservation attributes in the extract, no outcome column (block-size proxy)",
    "agent_id": "agent as string ('none' if null)",
    "arrival_month_num": "from arrival_date", "arrival_week_iso": "ISO week of arrival_date",
    "arrival_dow": "weekday of arrival_date", "total_nights": "weekend + week nights",
    "total_guests": "adults + children + babies", "adr_clean": "raw REAL adr; the data error (> 1000, one row) set to NULL",
    "gross_value": "adr_clean x total_nights", "is_duplicate": "exact duplicate flag (notebook 01)",
    "exclude_from_model": "data-quality exclusion flag (notebook 01)",
}
for c in bk.columns:
    if c in derived_detail:
        add("bookings", c, "DERIVED", derived_detail[c])
    else:
        add("bookings", c, "REAL", REAL_PMS)
for r in rows:
    r["use"] = ("candidate (NB01); final lists in notebook 03" if r["column"] in GROUPS["nb01_candidate_columns"] else
                "ops only (never a feature)" if r["column"] in GROUPS["ops_only_cols"] else "id / target")

add("dim_room_type", "room_type", "REAL", REAL_PMS + " (room codes)", "capacity allocation")
add("dim_room_type", "capacity_rooms", "DERIVED", "peak occupied rooms per type; assumptions: operations.room_capacity_by_type", "ops / scenarios only")
add("dim_room_type", "is_sellable", "DERIVED", "operations.non_sellable_room_types", "ops / scenarios only")
add("dim_room_type", "capacity_source", "DERIVED", "label of the capacity method", "documentation")
for t in ["cx_contact", "cx_feedback", "fin_booking"]:
    add(t, "booking_id", "DERIVED", "key from bookings.booking_id", "join key")
for col, stype, det in [
    ("contact_attempted", "SYNTHETIC", "cx.contact_probability; no contact if cancelled before planned date"),
    ("contact_channel", "SYNTHETIC", "cx.contact_channel_mix"),
    ("contact_date", "SYNTHETIC", "cx.contact_days_before_arrival"),
    ("customer_response", "SYNTHETIC", "cx.response_probability (outcome-conditioned -> never a feature)")]:
    add("cx_contact", col, stype, det, "follow-up design, response-rate KPIs; never a feature")
for col, det, use in [
    ("satisfaction_score", "cx.satisfaction (Check-Out only)", "CX analysis; never a feature"),
    ("cancellation_reason_category", "cx.cancellation_reason (Canceled only)", "text-analytics ground truth; never a feature"),
    ("cancellation_reason_text", "templates by category", "TF-IDF + clustering demo; never a feature")]:
    add("cx_feedback", col, "SYNTHETIC", det, use)
for col, det in [
    ("room_cost_per_night", "finance.rooms_cost_ratio x median ADR (room type, month)"),
    ("channel_commission_rate", "finance.commission_rate_by_segment / gds_commission_rate"),
    ("rebooking_revenue", "resale simulation (operations.resale_probability, resale_price_factor)"),
    ("rebooked_room_nights", "resale simulation (same draw as rebooking_revenue)"),
    ("cancellation_handling_cost", "finance.handling_minutes x staff_cost_per_hour (is_canceled = 1)")]:
    add("fin_booking", col, "SYNTHETIC", det, "net revenue, revenue at risk, scenarios; never a feature")
for col, stype, det in [
    ("stay_date", "DERIVED", "calendar of stay nights"), ("room_type", "REAL", "room codes"),
    ("available_rooms", "DERIVED", "dim_room_type.capacity_rooms"),
    ("rooms_sold", "DERIVED", "non-cancelled stay nights x assigned_room_type"),
    ("rooms_cancelled", "DERIVED", "cancelled stay nights x reserved_room_type (no-shows excluded)"),
    ("resold_after_cancellation", "SYNTHETIC", "resale simulation, capped by rooms_sold"),
    ("occupancy_rate", "DERIVED", "rooms_sold / available_rooms"),
    ("is_partial_period", "DERIVED", "date outside operations.room_capacity_by_type.window")]:
    add("ops_inventory_daily", col, stype, det, "capacity allocation, resale, price adjustment; never a feature")
# technical flag present in the four simulated tables
for t in ["cx_contact", "cx_feedback", "fin_booking", "ops_inventory_daily"]:
    add(t, "is_synthetic", "SYNTHETIC" if t != "ops_inventory_daily" else "DERIVED + SYNTHETIC",
        "constant TRUE set by notebook 02", "labelling only (presentation / audit); never a feature")
# every column of the SYSTEM tables in sql/schema.sql (checked against the DDL below)
SYSTEM_COLS = {
    "risk_scores": ["booking_id", "scored_at", "model_version", "p_cancel", "risk_tier", "expected_revenue_at_risk", "days_to_arrival"],
    "action_plan": ["action_id", "booking_id", "stay_date", "room_type", "action_type", "rule_id", "needs_approval",
                    "approval_status", "approved_by", "approved_at", "created_at"],
    "pilot_assignment": ["pilot_id", "booking_id", "arm", "stratum", "assigned_at"],
    "interventions": ["intervention_id", "action_id", "channel", "executed_at", "executed_by", "outcome", "cost"],
}
SYSTEM_SRC = {"risk_scores": "ML model (later stage)", "action_plan": "rule engine + staff approval (later stage)",
              "pilot_assignment": "pilot randomisation (later stage)", "interventions": "n8n / staff execution log"}
for t, cols in SYSTEM_COLS.items():
    for c in cols:
        add(t, c, "SYSTEM", SYSTEM_SRC[t], "written by later stages")
import re as _re
_ddl = (ROOT / "sql" / "schema.sql").read_text(encoding="utf-8")
DDL_COLS = {}
for t in _re.findall(r"CREATE TABLE ([a-z_]+) \(", _ddl):
    body = _re.search(r"CREATE TABLE " + t + r" \((.*?)\n\);", _ddl, _re.S).group(1)
    DDL_COLS[t] = [m.group(1) for m in _re.finditer(r"^\s+([a-z_]+)\s+[A-Z]", body, _re.M) if m.group(1) not in ("PRIMARY", "CHECK", "FOREIGN")]
for t, cols in SYSTEM_COLS.items():
    assert DDL_COLS[t] == cols, (t, DDL_COLS[t], cols)

# ---- Meaning, unit/domain and timing of every field ------------------------------------------
# needed_at    = when the business decision would need the value
# in_dataset_as = what this extract actually stores. The public extract is a near-arrival snapshot, so a value
#                 the business needs at booking entry is usually NOT stored as its booking-time value.
ENTRY, OUTCOME, LATER = "booking entry", "after the outcome (never a feature)", "n/a (system field)"
SNAP = "near-arrival snapshot; may have been revised after booking; booking-time value not verified"
STAY = "planned stay as last recorded in the near-arrival snapshot; original booking-time value not verified"
CAL = "deterministic transform of the snapshot arrival date; not proof the arrival date was unchanged since booking"
FINAL = "final value after the outcome"
FIXED = "fixed / derived (no timing)"
SIM_STATIC = "simulated from config/assumptions.yaml (scenario input, not observed)"
SIM_OUT = "simulated from config/assumptions.yaml; depends on the outcome (never a feature)"
META = {  # column: (description, unit or domain, needed_at, in_dataset_as)
    "booking_id": ("Surrogate booking key, 1..79,330 in file order after filtering to City Hotel", "integer", FIXED, FIXED),
    "is_canceled": ("Target: 1 if the booking was cancelled or was a no-show, 0 if the guest checked in", "0/1", "label (after the outcome)", OUTCOME),
    "reservation_status": ("Final status of the booking", "Check-Out / Canceled / No-Show", "after the outcome", OUTCOME),
    "reservation_status_date": ("Date of the last status (cancellation date or check-out date)", "date", "after the outcome", OUTCOME),
    "booking_date": ("Booking entry date reconstructed as arrival_date - lead_time", "date", ENTRY,
                     "computed from snapshot arrival_date and lead_time; not a recorded booking timestamp"),
    "arrival_date": ("Planned arrival date (year + month + day columns)", "date", ENTRY, STAY),
    "lead_time": ("Days between booking entry and planned arrival", "days, >= 0", ENTRY,
                  "as last recorded in the snapshot; a revised arrival date would change it (history not available)"),
    "stays_in_weekend_nights": ("Booked Saturday/Sunday nights", "nights", ENTRY, STAY),
    "stays_in_week_nights": ("Booked Monday-Friday nights", "nights", ENTRY, STAY),
    "adults": ("Number of adults", "count", ENTRY, SNAP),
    "children": ("Number of children (4 missing values filled with 0)", "count", ENTRY, SNAP),
    "babies": ("Number of babies", "count", ENTRY, SNAP),
    "meal": ("Meal package booked", "BB / HB / FB / SC", ENTRY, SNAP),
    "country": ("Country of origin (ISO 3166-1 alpha-3; UNK if missing)", "code", ENTRY,
                "value in the extract; booking-time value not verified in this extract (strong association with the outcome)"),
    "market_segment": ("Market segment designation", "Online TA, Offline TA/TO, Groups, Direct, Corporate, Aviation, Complementary, Undefined", ENTRY, SNAP),
    "distribution_channel": ("Booking distribution channel", "TA/TO, Direct, Corporate, GDS, Undefined", ENTRY, SNAP),
    "is_repeated_guest": ("1 if the guest had stayed before", "0/1", ENTRY, SNAP),
    "previous_cancellations": ("Number of the guest's earlier bookings that were cancelled", "count", ENTRY, SNAP),
    "previous_bookings_not_canceled": ("Number of the guest's earlier bookings that were not cancelled", "count", ENTRY, SNAP),
    "reserved_room_type": ("Room type reserved (anonymised code)", "A..P", ENTRY, SNAP),
    "assigned_room_type": ("Room type actually assigned; may differ from reserved", "A..P", "check-in",
                           "final value; allocation time unknown (may change until check-in)"),
    "booking_changes": ("Number of changes made from booking entry until check-in or cancellation (cumulative)", "count", "n/a (excluded)",
                        "cumulative count up to check-in or cancellation: contains future information"),
    "deposit_type": ("Deposit condition of the booking", "No Deposit / Non Refund / Refundable", ENTRY, SNAP),
    "days_in_waiting_list": ("Days the booking waited before being confirmed", "days", "pre-arrival horizon",
                             "completed waiting duration in the snapshot"),
    "customer_type": ("Type of booking", "Transient / Transient-Party / Contract / Group", ENTRY, SNAP),
    "total_of_special_requests": ("Number of special requests (e.g. twin bed, high floor); can be added after booking", "count",
                                  "pre-arrival horizon", "count in the near-arrival snapshot (may include requests added later)"),
    "has_agent": ("1 if the booking came through a travel agent", "0/1", ENTRY, SNAP),
    "has_company": ("1 if a company/entity is responsible for payment", "0/1", ENTRY, SNAP),
    "dup_group_size": ("Number of bookings sharing the same 21 reservation attributes in the extract (block-size proxy; not a verified group ID)",
                       "count >= 1", ENTRY + " (or end of booking day)",
                       "retrospective group size from snapshot attributes; availability at booking entry or end of booking day is not verified"),
    "agent_id": ("Anonymised travel-agent ID; 'none' if no agent", "code", ENTRY, SNAP),
    "arrival_month_num": ("Arrival month", "1..12", ENTRY, CAL),
    "arrival_week_iso": ("ISO-8601 week of arrival_date", "1..53", ENTRY, CAL),
    "arrival_dow": ("Weekday of arrival", "Monday..Sunday", ENTRY, CAL),
    "total_nights": ("stays_in_weekend_nights + stays_in_week_nights", "nights", ENTRY, STAY),
    "total_guests": ("adults + children + babies", "count", ENTRY, SNAP),
    "adr_clean": ("Average daily rate (room revenue / nights); the EUR 5,400 data error set to NULL", "EUR per night", ENTRY,
                  "near-arrival, transaction-derived value; the rate quoted at booking is not available"),
    "gross_value": ("Gross booked room revenue: adr_clean x total_nights (before commission, cost, resale)", "EUR", ENTRY, "inherits adr_clean and total_nights (snapshot)"),
    "is_duplicate": ("1 if the row is an exact copy of an earlier row on all 32 original columns (data-quality flag; key includes outcome)", "0/1", "n/a (quality flag)", FINAL),
    "exclude_from_model": ("True if the row fails a data-quality rule and is excluded from modelling", "bool", FIXED, FIXED),
    "room_type": ("Room type code", "A..P", FIXED, FIXED),
    "capacity_rooms": ("Assumed sellable rooms of this type = peak occupied rooms Sep-2015..Aug-2017 (uses realised 2017 stays)", "rooms", FIXED,
                       "derived from realised stays incl. the 2017 test period (ops only)"),
    "is_sellable": ("False for room type P (no stay nights, ADR always 0)", "bool", FIXED, FIXED),
    "capacity_source": ("How capacity_rooms was derived", "text", FIXED, FIXED),
    "contact_attempted": ("Simulated pre-arrival contact happened (not if the booking was cancelled first)", "bool", "contact date", SIM_OUT),
    "contact_channel": ("Simulated contact channel", "email / sms / phone", "contact date", SIM_OUT),
    "contact_date": ("Simulated contact date (between booking and arrival)", "date", "contact date", SIM_OUT),
    "customer_response": ("Simulated response, drawn conditionally on the outcome", "confirmed / requested_change / no_response", "after contact", SIM_OUT),
    "satisfaction_score": ("Simulated post-stay satisfaction (Check-Out only)", "1..5", "after check-out", SIM_OUT),
    "cancellation_reason_category": ("Simulated cancellation reason category (Canceled only)", "7 categories incl. not_provided", "after cancellation", SIM_OUT),
    "cancellation_reason_text": ("Simulated free-text reason from templates (Canceled only)", "text", "after cancellation", SIM_OUT),
    "room_cost_per_night": ("Simulated cost of servicing one occupied room-night", "EUR per night", "decision time (finance input)", SIM_STATIC),
    "channel_commission_rate": ("Assumed commission share paid to the channel", "0..1", "decision time (finance input)", SIM_STATIC),
    "rebooking_revenue": ("Simulated GROSS revenue from reselling the cancelled room-nights (Canceled only; 0 if not resold)", "EUR", "after resale", SIM_OUT),
    "rebooked_room_nights": ("Simulated number of the booking's cancelled room-nights that were resold (Canceled only)", "room-nights", "after resale", SIM_OUT),
    "cancellation_handling_cost": ("Simulated staff cost of processing a cancellation (is_canceled = 1 only)", "EUR", "after cancellation", SIM_OUT),
    "stay_date": ("Calendar night", "date", FIXED, FIXED),
    "available_rooms": ("Sellable rooms of this type on this night (= capacity_rooms)", "rooms", "decision time (ops input)",
                        "derived from realised stays incl. 2017 (ops only)"),
    "rooms_sold": ("Occupied rooms: non-cancelled stay nights by assigned_room_type", "rooms", "after the stay", "realised outcome (ops only)"),
    "rooms_cancelled": ("Cancelled stay nights by reserved_room_type (no-shows excluded)", "rooms", "after cancellation", "realised outcome (ops only)"),
    "resold_after_cancellation": ("Simulated cancelled room-nights that were resold (<= rooms_sold, <= rooms_cancelled)", "rooms", "after resale", SIM_OUT),
    "occupancy_rate": ("rooms_sold / available_rooms", "0..1", "after the stay", "realised outcome (ops only)"),
    "is_partial_period": ("True for nights outside 2015-09-01..2017-08-31 (incomplete coverage)", "bool", FIXED, FIXED),
    "scored_at": ("Timestamp of the score", "timestamp", "score time", LATER),
    "model_version": ("Identifier of the model that produced the score", "text", "score time", LATER),
    "days_to_arrival": ("arrival_date minus score date", "days", "score time", LATER),
    "approved_by": ("Staff member who approved or rejected the action", "text", "approval time", LATER),
    "approved_at": ("Timestamp of the approval or rejection (source of KPI P3)", "timestamp", "approval time", LATER),
    "created_at": ("Timestamp when the rule engine proposed the action", "timestamp", "decision time", LATER),
    "pilot_id": ("Identifier of the pilot / experiment", "text", "assignment time", LATER),
    "arm": ("Randomised arm: treatment (rules applied) or holdout (business as usual)", "treatment / holdout", "assignment time", LATER),
    "stratum": ("Stratum used for stratified randomisation (e.g. risk tier, population)", "text", "assignment time", LATER),
    "assigned_at": ("Timestamp of the random assignment", "timestamp", "assignment time", LATER),
    "p_cancel": ("Calibrated cancellation probability written by the model", "0..1", "score time", LATER),
    "risk_tier": ("Risk tier derived from p_cancel", "low / medium / high", "score time", LATER),
    "expected_revenue_at_risk": ("p_cancel x booking value", "EUR", "score time", LATER),
    "action_id": ("Surrogate key of a proposed action", "integer", "decision time", LATER),
    "action_type": ("Proposed action", "see schema CHECK list", "decision time", LATER),
    "rule_id": ("Rule that proposed the action", "text", "decision time", LATER),
    "needs_approval": ("Whether staff approval is required before execution", "bool", "decision time", LATER),
    "approval_status": ("Approval state", "auto / pending / approved / rejected", "decision time", LATER),
    "intervention_id": ("Surrogate key of an executed intervention", "integer", "execution", LATER),
    "channel": ("Execution channel", "email / sms / phone / pms / task", "execution", LATER),
    "executed_at": ("Execution timestamp", "timestamp", "execution", LATER),
    "executed_by": ("Executor", "n8n / staff", "execution", LATER),
    "outcome": ("Observed result of the intervention", "text", "after execution", LATER),
    "cost": ("Cost of the intervention", "EUR", "after execution", LATER),
}

# ---- logical dtype and null rule, read from the tables actually built above ------------------
TABLES = {"bookings": bk, "dim_room_type": dim_room_type, "cx_contact": cx_contact, "cx_feedback": cx_feedback,
          "fin_booking": fin, "ops_inventory_daily": ops_out}
def logical_dtype(s):
    if pd.api.types.is_bool_dtype(s): return "boolean"
    if pd.api.types.is_integer_dtype(s): return "integer"
    if pd.api.types.is_float_dtype(s): return "decimal"
    if pd.api.types.is_datetime64_any_dtype(s): return "date"
    first = s.dropna().iloc[0] if s.notna().any() else None
    return "date" if hasattr(first, "isoformat") else "text"
SYSTEM_DTYPE = {"booking_id": "integer", "scored_at": "timestamp", "model_version": "text", "days_to_arrival": "integer",
                "stay_date": "date", "room_type": "text", "approved_by": "text", "approved_at": "timestamp", "created_at": "timestamp",
                "pilot_id": "text", "arm": "text", "stratum": "text", "assigned_at": "timestamp", "p_cancel": "decimal", "risk_tier": "text",
                "expected_revenue_at_risk": "decimal", "action_id": "integer", "action_type": "text", "rule_id": "text",
                "needs_approval": "boolean", "approval_status": "text", "intervention_id": "integer", "channel": "text",
                "executed_at": "timestamp", "executed_by": "text", "outcome": "text", "cost": "decimal"}
NULL_RULES = {  # (table, column) -> rule, for every column that can be NULL
    ("bookings", "adr_clean"): "NULL only for the one ADR data-error row (excluded from the model)",
    ("cx_contact", "contact_channel"): "NULL unless contact_attempted", ("cx_contact", "contact_date"): "NULL unless contact_attempted",
    ("cx_contact", "customer_response"): "NULL unless contact_attempted",
    ("cx_feedback", "satisfaction_score"): "NULL unless reservation_status = Check-Out",
    ("cx_feedback", "cancellation_reason_category"): "NULL unless reservation_status = Canceled",
    ("cx_feedback", "cancellation_reason_text"): "NULL unless Canceled and a reason was provided (category != not_provided)",
    ("fin_booking", "rebooking_revenue"): "NULL unless reservation_status = Canceled (0 if not resold)",
    ("fin_booking", "rebooked_room_nights"): "NULL unless reservation_status = Canceled (0 if not resold)",
    ("fin_booking", "cancellation_handling_cost"): "NULL unless is_canceled = 1",
    ("ops_inventory_daily", "occupancy_rate"): "NULL when available_rooms = 0 (room type P)",
    ("risk_scores", "risk_tier"): "nullable", ("risk_scores", "expected_revenue_at_risk"): "nullable",
    ("risk_scores", "days_to_arrival"): "nullable",
    ("action_plan", "booking_id"): "NULL for date x room-type actions", ("action_plan", "stay_date"): "NULL for booking-level actions",
    ("action_plan", "room_type"): "NULL for booking-level actions",
    ("action_plan", "approved_by"): "NOT NULL exactly when approval_status is approved / rejected",
    ("action_plan", "approved_at"): "NOT NULL exactly when approval_status is approved / rejected",
    ("pilot_assignment", "stratum"): "nullable",
    ("interventions", "channel"): "nullable", ("interventions", "outcome"): "NULL until the result is known",
    ("interventions", "cost"): "nullable",
}
KEYS = {
    ("bookings", "booking_id"): "PK", ("bookings", "reserved_room_type"): "FK -> dim_room_type.room_type",
    ("bookings", "assigned_room_type"): "FK -> dim_room_type.room_type", ("dim_room_type", "room_type"): "PK",
    ("cx_contact", "booking_id"): "PK, FK -> bookings.booking_id", ("cx_feedback", "booking_id"): "PK, FK -> bookings.booking_id",
    ("fin_booking", "booking_id"): "PK, FK -> bookings.booking_id",
    ("ops_inventory_daily", "stay_date"): "PK (stay_date, room_type)",
    ("ops_inventory_daily", "room_type"): "PK (stay_date, room_type); FK -> dim_room_type.room_type",
    ("risk_scores", "booking_id"): "PK (booking_id, scored_at); FK -> bookings.booking_id",
    ("risk_scores", "scored_at"): "PK (booking_id, scored_at)",
    ("action_plan", "action_id"): "PK", ("interventions", "intervention_id"): "PK",
    ("action_plan", "booking_id"): "FK -> bookings.booking_id", ("action_plan", "stay_date"): "FK (stay_date, room_type) -> ops_inventory_daily",
    ("action_plan", "room_type"): "FK -> dim_room_type.room_type; FK (stay_date, room_type) -> ops_inventory_daily",
    ("pilot_assignment", "pilot_id"): "PK (pilot_id, booking_id)",
    ("pilot_assignment", "booking_id"): "PK (pilot_id, booking_id); FK -> bookings.booking_id",
    ("interventions", "action_id"): "FK -> action_plan.action_id",
}
META_BY_TABLE = {  # (table, column) overrides where the same column name means something table-specific
    ("cx_contact", "is_synthetic"): ("Flag: every row of this table is simulated (always TRUE)", "bool", FIXED, SIM_STATIC),
    ("cx_feedback", "is_synthetic"): ("Flag: every row of this table is simulated (always TRUE)", "bool", FIXED, SIM_STATIC),
    ("fin_booking", "is_synthetic"): ("Flag: every row of this table is simulated (always TRUE)", "bool", FIXED, SIM_STATIC),
    ("ops_inventory_daily", "is_synthetic"): ("Flag (always TRUE): the row mixes DERIVED fields (capacity, rooms sold / cancelled, occupancy) "
                                              "with the SYNTHETIC resold_after_cancellation", "bool", FIXED, "derived + simulated (assumptions.yaml)"),
}
for r in rows:
    t, c = r["table"], r["column"]
    r["description"], r["unit_or_domain"], r["needed_at"], r["in_dataset_as"] = META_BY_TABLE.get((t, c), META.get(c, ("", "", "", "")))
    if t in TABLES:
        s = TABLES[t][c]
        r["dtype"] = logical_dtype(s)
        has_null = bool(s.isna().any())
        r["null_rule"] = NULL_RULES.get((t, c), "never null")
        assert not has_null or (t, c) in NULL_RULES, f"{t}.{c} has NULLs but no documented rule"
    else:
        r["dtype"] = SYSTEM_DTYPE[c]
        r["null_rule"] = NULL_RULES.get((t, c), "never null")
    r["key"] = KEYS.get((t, c), "")
missing = [f'{r["table"]}.{r["column"]}' for r in rows if not r["description"]]
assert not missing, missing
listed = {(r["table"], r["column"]) for r in rows}
not_listed = [f"{t}.{c}" for t, d in TABLES.items() for c in d.columns if (t, c) not in listed]
assert not not_listed, not_listed
ddl_set = {(t, c) for t, cols in DDL_COLS.items() for c in cols}
assert listed == ddl_set, ("dictionary vs schema.sql", sorted(listed ^ ddl_set))   # every column of every SQL table, no extras

data_dict = pd.DataFrame(rows)[["table", "column", "source_type", "source_detail", "description", "unit_or_domain",
                                "dtype", "null_rule", "key", "needed_at", "in_dataset_as", "use"]]
data_dict.to_csv(DOCS_DIR / "data_dictionary.csv", index=False)
print(f"data_dictionary.csv: {len(data_dict)} fields x {data_dict.shape[1]} attributes")
pd.crosstab(data_dict.table, data_dict.source_type)


data_dictionary.csv: 100 fields x 12 attributes


source_type,DERIVED,DERIVED + SYNTHETIC,REAL,SYNTHETIC,SYSTEM
table,,,,,
action_plan,0,0,0,0,11
bookings,16,0,23,0,0
cx_contact,1,0,0,5,0
cx_feedback,1,0,0,4,0
dim_room_type,3,0,1,0,0
fin_booking,1,0,0,6,0
interventions,0,0,0,0,7
ops_inventory_daily,6,1,1,1,0
pilot_assignment,0,0,0,0,5


## 6. Example data extract (real + simulated, clearly marked)

A small, **stratified** extract (30 bookings) that shows every case: stayed, cancelled and resold, cancelled and not resold, no-show, contacted and not contacted. Column prefixes mark the origin:

| Prefix | Meaning |
|---|---|
| `real__` | from the hotel PMS dataset |
| `drv__` | derived only from real data |
| `syn__` | **simulated** from `config/assumptions.yaml` |

In [14]:
full = (bk.merge(cx_contact.drop(columns="is_synthetic"), on="booking_id")
          .merge(cx_feedback.drop(columns="is_synthetic"), on="booking_id")
          .merge(fin.drop(columns="is_synthetic"), on="booking_id"))
full["resold_any"] = full.rebooking_revenue.fillna(0) > 0

strata = {
    "stayed, contacted": (full.is_canceled == 0) & full.contact_attempted,
    "stayed, not contacted": (full.is_canceled == 0) & ~full.contact_attempted,
    "cancelled, resold": full.reservation_status.eq("Canceled") & full.resold_any,
    "cancelled, not resold": full.reservation_status.eq("Canceled") & ~full.resold_any,
    "no-show": full.reservation_status.eq("No-Show"),
}
n_per = {"stayed, contacted": 6, "stayed, not contacted": 6, "cancelled, resold": 7,
         "cancelled, not resold": 7, "no-show": 4}
pick = pd.concat([full[m & ~full.exclude_from_model].sample(n_per[k], random_state=A["meta"]["sample_extract_seed"]).assign(case=k)
                  for k, m in strata.items()])

real_cols = ["is_canceled", "reservation_status", "reservation_status_date",
             "lead_time", "market_segment", "distribution_channel", "deposit_type", "customer_type",
             "reserved_room_type", "assigned_room_type", "total_of_special_requests"]
drv_cols = ["booking_id", "arrival_date", "booking_date", "total_nights", "adr_clean", "gross_value"]
syn_cols = ["contact_attempted", "contact_channel", "contact_date", "customer_response",
            "satisfaction_score", "cancellation_reason_category", "cancellation_reason_text",
            "room_cost_per_night", "channel_commission_rate", "cancellation_handling_cost", "rebooking_revenue"]
extract = pick[["case"] + drv_cols[:2] + real_cols + drv_cols[2:] + syn_cols].rename(
    columns={**{c: f"real__{c}" for c in real_cols}, **{c: f"drv__{c}" for c in drv_cols},
             **{c: f"syn__{c}" for c in syn_cols}})
extract.to_csv(SYN_DIR / "sample_extract_bookings.csv", index=False, date_format="%Y-%m-%d")
extract

,case,drv__booking_id,drv__arrival_date,real__is_canceled,real__reservation_status,real__reservation_status_date,real__lead_time,real__market_segment,real__distribution_channel,real__deposit_type,real__customer_type,real__reserved_room_type,real__assigned_room_type,real__total_of_special_requests,drv__booking_date,drv__total_nights,drv__adr_clean,drv__gross_value,syn__contact_attempted,syn__contact_channel,syn__contact_date,syn__customer_response,syn__satisfaction_score,syn__cancellation_reason_category,syn__cancellation_reason_text,syn__room_cost_per_night,syn__channel_commission_rate,syn__cancellation_handling_cost,syn__rebooking_revenue
4971,"stayed, contacted",4972,2015-10-16,0,Check-Out,2015-10-18,102,Offline TA/TO,TA/TO,No Deposit,Transient-Party,A,A,0,2015-07-06,2,109.00,218.00,True,sms,2015-10-02,confirmed,3,NaN,NaN,23.45,0.20,NaN,NaN
55513,"stayed, contacted",55514,2016-08-14,0,Check-Out,2016-08-21,162,Online TA,TA/TO,No Deposit,Transient,A,A,1,2016-03-05,7,120.82,845.74,True,sms,2016-07-31,confirmed,3,NaN,NaN,26.21,0.18,NaN,NaN
64990,"stayed, contacted",64991,2017-01-07,0,Check-Out,2017-01-29,82,Online TA,TA/TO,No Deposit,Transient,A,A,4,2016-10-17,22,74.80,1645.60,True,phone,2016-12-31,requested_change,4,NaN,NaN,22.14,0.18,NaN,NaN
63315,"stayed, contacted",63316,2016-12-19,0,Check-Out,2016-12-21,39,Direct,Direct,No Deposit,Transient,A,A,2,2016-11-10,2,38.00,76.00,True,email,2016-12-16,confirmed,4,NaN,NaN,19.59,0.00,NaN,NaN
52857,"stayed, contacted",52858,2016-07-11,0,Check-Out,2016-07-14,110,Online TA,TA/TO,No Deposit,Transient,A,A,0,2016-03-23,3,108.30,324.90,True,email,2016-06-27,confirmed,5,NaN,NaN,25.34,0.18,NaN,NaN
53984,"stayed, contacted",53985,2016-07-29,0,Check-Out,2016-07-30,164,Online TA,TA/TO,No Deposit,Transient,A,A,0,2016-02-16,1,105.30,105.30,True,sms,2016-07-15,confirmed,3,NaN,NaN,26.59,0.18,NaN,NaN
62322,"stayed, not contacted",62323,2016-11-24,0,Check-Out,2016-11-27,112,Online TA,TA/TO,No Deposit,Transient,A,A,1,2016-08-04,3,90.00,270.00,False,None,None,None,5,NaN,NaN,24.15,0.18,NaN,NaN
50942,"stayed, not contacted",50943,2016-06-14,0,Check-Out,2016-06-15,11,Direct,Direct,No Deposit,Transient,A,A,0,2016-06-03,1,91.00,91.00,False,None,None,None,4,NaN,NaN,28.58,0.00,NaN,NaN
50152,"stayed, not contacted",50153,2016-06-03,0,Check-Out,2016-06-04,255,Online TA,TA/TO,No Deposit,Transient,A,C,1,2015-09-22,1,80.85,80.85,False,None,None,None,4,NaN,NaN,32.48,0.18,NaN,NaN
36672,"stayed, not contacted",36673,2015-08-04,0,Check-Out,2015-08-06,1,Corporate,Direct,No Deposit,Transient,A,A,0,2015-08-03,2,65.00,130.00,False,None,None,None,4,NaN,NaN,25.64,0.00,NaN,NaN


## 7. Export tables and load into PostgreSQL

In [15]:
dim_room_type.to_csv(SYN_DIR / "dim_room_type.csv", index=False)
cx_contact.to_csv(SYN_DIR / "cx_contact.csv", index=False)
cx_feedback.to_csv(SYN_DIR / "cx_feedback.csv", index=False)
fin.to_csv(SYN_DIR / "fin_booking.csv", index=False)
ops_out[["stay_date", "room_type", "available_rooms", "rooms_sold", "rooms_cancelled",
         "resold_after_cancellation", "occupancy_rate", "is_partial_period", "is_synthetic"]] \
    .to_csv(SYN_DIR / "ops_inventory_daily.csv", index=False, date_format="%Y-%m-%d")

for f in sorted(SYN_DIR.glob("*.csv")):
    print(f"{f.name:32s} {sum(1 for _ in open(f)) - 1:>8,} rows")

cx_contact.csv                     79,330 rows
cx_feedback.csv                    79,330 rows
dim_room_type.csv                       9 rows
fin_booking.csv                    79,330 rows
ops_inventory_daily.csv             7,227 rows
sample_extract_bookings.csv            30 rows


To load everything into PostgreSQL (from the repo root):

```bash
createdb hotel_dss
psql -d hotel_dss -f sql/schema.sql
psql -d hotel_dss -f sql/load.sql
```

`sql/load.sql` uses `\copy`, so it reads the CSVs from the client machine. Load order respects the foreign keys: `dim_room_type` → `bookings` → booking-level tables → `ops_inventory_daily`. Assumptions stay in `config/assumptions.yaml` (not loaded).

## 8. Integration and privacy considerations

### 8.1 Integration

| Topic | Issue in a real deployment | How the design handles it |
|---|---|---|
| **Source systems** | Bookings come from the PMS; OTA bookings via a channel manager; contacts from email/SMS tools; costs from accounting | One PostgreSQL schema with the PMS booking ID as the master key; each source writes its own table |
| **Keys** | The public dataset has no booking ID; in production, PMS and channel-manager IDs differ | `booking_id` is the master key; add a mapping table (`pms_id`, `channel_ref`) when connecting real sources |
| **Different grains** | Booking-level (CX, finance) vs date × room type (inventory) | Separate tables; join through stay nights; views `v_booking_360`, `v_ops_daily_hotel` |
| **Timing / snapshot** | The dataset is a day-before-arrival snapshot; live scoring needs the booking state *at decision time* | System tables are time-stamped (`scored_at`, `created_at`, `executed_at`); production should store booking-change history (SCD type 2) |
| **Real-time inventory** | Capacity and rooms sold change continuously | `ops_inventory_daily` updated by n8n from the PMS/channel manager (prototype: batch refresh) |
| **Data quality** | Duplicates, zero-night and zero-guest rows, ADR errors | Flags from notebook 01 (`is_duplicate`, `exclude_from_model`) travel with the data |
| **Synthetic vs real** | Risk of presenting simulated numbers as facts | `is_synthetic` column; `syn_` prefix in views and extracts; every parameter traceable in `assumptions` |
| **Automation loop** | n8n must not act without control | `action_plan.needs_approval` / `approval_status` record the decision; every execution is logged in `interventions`; a guard that refuses execution of pending/rejected actions is a G2 requirement |

### 8.2 Privacy and ethics

- **This dataset:** anonymised by the authors. There are no names, contacts or payment data; `agent` and `company` are IDs. Analysis is low-risk.
- **A real deployment processes personal data** (name, email, phone, nationality, stay dates), so **GDPR** applies (Portugal / EU; supervisory authority: CNPD):
  - *Lawful basis:* booking confirmations and reminders are part of performing the contract. Promotional offers need **consent** and an opt-out.
  - *Data minimisation:* the risk model doesn't need names or contacts. Keep PII in a separate, access-controlled table keyed by `booking_id`, and give analysts and the dashboard pseudonymised data.
  - *Retention:* delete or anonymise contact data after the stay plus the statutory period; keep aggregated KPIs only.
  - *Automated decisions (Art. 22):* actions with significant effects on a guest (deposit requests, cancellation by the hotel, overbooking relocation) need **human approval**. The schema **records** this (`needs_approval`, `approval_status`, `approved_by`); an execution guard that blocks unapproved high-impact actions is specified for the automation stage (G2).
  - *Transparency:* tell guests in the privacy notice that bookings are risk-scored to plan follow-ups.
- **Fairness:** don't use `country` / nationality to treat guests differently. It is excluded from the model features (its booking-time value is also not verified in this extract). Monitor actions by segment to avoid systematically burdening some guests (e.g. extra deposits).
- **Security:** role-based access (front office vs revenue manager vs analyst), encrypted credentials for n8n integrations, and an audit trail via `interventions`.
- **Synthetic data disclosure:** the report and dashboard must state which figures are simulated and from which assumptions.